## Silver Layer

In [1]:
!pip install duckdb

In [12]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import boto3
import pandas as pd
import numpy as np
import duckdb as db
import io

In [54]:
bucket_name = "airbnb-pipeline-data"
file_key = "bronze/listings/listings.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket_name, Key=file_key)

listings_br = pd.read_csv(obj['Body'])

listings_br.head(2)

,listing_id,host_id,property_type,room_type,city,country,accommodates,bedrooms,bathrooms,price_per_night,created_at
0,1,150,House,Entire home,West Lisafort,Malta,3,1,2,84,2025-12-26 14:15:54.011160
1,2,91,Apartment,Private room,North Cherylberg,Western Sahara,1,2,3,140,2025-12-26 14:15:54.011160


In [55]:
listings_br = listings_br.drop_duplicates()

listings_br = listings_br.fillna({
    "property_type":"Unknown",
    "room_type":"Unknown",
    "city":"Unknown",
    "country":"Unknown",
    "accommodates": 0,
    "bedrooms": 0,
    "bathrooms": 0,
    "price_per_night": 0
})

listings_br['modified_at'] = pd.Timestamp.now(tz='UTC')

listings_sl = listings_br

In [56]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/listings/listings.csv"

csv_buffer = io.StringIO()
listings_sl.to_csv(csv_buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name,Key=file_key,Body=csv_buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [57]:
bucket_name = "airbnb-pipeline-data"
file_key = "bronze/bookings/bookings.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket = bucket_name, Key = file_key)

bookings_br = pd.read_csv(obj['Body'])

bookings_br.head(2)

,booking_id,listing_id,booking_date,nights_booked,booking_amount,cleaning_fee,service_fee,booking_status,created_at
0,ef0bd262-f88d-477f-aeba-9c69cc72c051,379,2024-12-25,3,591,70,23,confirmed,2025-12-26 14:15:54.011160
1,490685e5-1b70-429b-84ac-d605509467af,111,2025-02-18,3,171,45,19,confirmed,2025-12-26 14:15:54.011160


In [58]:
bookings_br = bookings_br.drop_duplicates()

bookings_br = bookings_br.fillna({  
    "booking_status":"Unknown",
    "nights_booked": 0,
    "booking_amount": 0,
    "cleaning_fee": 0,
    "service_fee": 0

})

bookings_br['modified_at'] = pd.Timestamp.now(tz='UTC')

bookings_sl = bookings_br

In [59]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/bookings/bookings.csv"

csv_buffer = io.StringIO()
bookings_sl.to_csv(csv_buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name,Key=file_key,Body=csv_buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [60]:
bucket_name = "airbnb-pipeline-data"
file_key = "bronze/hosts/hosts.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket = bucket_name, Key = file_key)

hosts_br = pd.read_csv(obj['Body'])

hosts_br.head(2)

,host_id,host_name,host_since,is_superhost,response_rate,created_at
0,1,Timothy Parker,2018-03-20,False,99,2025-12-26 14:15:54.011160
1,2,Hannah Evans,2024-01-01,False,95,2025-12-26 14:15:54.011160


In [61]:
hosts_br = hosts_br.drop_duplicates()

hosts_br = hosts_br.fillna({  
    "host_name":"Name Missing",
    "response_rate": 0
})

hosts_br['modified_at'] = pd.Timestamp.now(tz='UTC')

hosts_sl = hosts_br

In [62]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/hosts/hosts.csv"

csv_buffer = io.StringIO()
hosts_sl.to_csv(csv_buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name,Key=file_key,Body=csv_buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


## Gold Layer

In [3]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/listings/listings.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket_name,Key=file_key)

listings_sl = pd.read_csv(obj['Body'])

In [4]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/bookings/bookings.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket_name,Key=file_key)

bookings_sl = pd.read_csv(obj['Body'])

In [5]:
bucket_name = "airbnb-pipeline-data"
file_key = "silver/hosts/hosts.csv"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket_name,Key=file_key)

hosts_sl = pd.read_csv(obj['Body'])

In [6]:
bl_sl = pd.merge(bookings_sl, listings_sl, on='listing_id', how='left')

bl_sl = bl_sl[['booking_id', 'listing_id', 'booking_date', 'nights_booked',
       'booking_amount', 'cleaning_fee', 'service_fee', 'booking_status',
        'host_id', 'property_type','room_type', 'city', 'country', 'accommodates', 'bedrooms', 'bathrooms',
       'price_per_night']]

obt = pd.merge(bl_sl, hosts_sl, on ='host_id', how='left')

obt = obt[['booking_id', 'listing_id', 'booking_date', 'nights_booked',
       'booking_amount', 'cleaning_fee', 'service_fee', 'booking_status',
       'host_id', 'property_type', 'room_type', 'city', 'country',
       'accommodates', 'bedrooms', 'bathrooms', 'price_per_night', 'host_name',
       'host_since', 'is_superhost', 'response_rate']]

obt['created_at'] = pd.Timestamp.now(tz='UTC')

In [7]:
obt.columns

Index(['booking_id', 'listing_id', 'booking_date', 'nights_booked',
       'booking_amount', 'cleaning_fee', 'service_fee', 'booking_status',
       'host_id', 'property_type', 'room_type', 'city', 'country',
       'accommodates', 'bedrooms', 'bathrooms', 'price_per_night', 'host_name',
       'host_since', 'is_superhost', 'response_rate', 'created_at'],
      dtype='str')

In [8]:
obt['accommodation_type'] = np.where(obt['accommodates'] > 4,'Large',
                                     np.where(obt['accommodates'] > 2, 'Medium','Single')
                                    )

obt['total_amount'] = (obt['nights_booked']*obt['price_per_night'])+ obt['cleaning_fee'] + obt['service_fee']

obt['response_rate_quality'] = np.where(obt['response_rate'] > 90,'Very Good',
                                     np.where(obt['response_rate'] > 80, 'Good',
                                     np.where(obt['response_rate'] > 60, 'Average',
                                     'Poor')
                                    ))

In [9]:
obt['booking_date'] = pd.to_datetime(obt['booking_date'])

In [10]:
booking_analytics = """ 
with cr as (select 
year(booking_date) as year,
quarter(booking_date) as quarter,
month(booking_date) as month,
count(distinct booking_id) as total_bookings,
count(distinct host_id) as total_hosts,
count(distinct listing_id) as total_listing,
sum(accommodates) as total_accommodates,
sum(case when booking_status = 'cancelled' then 1 else 0 end) as total_cancelled
from obt
group by year(booking_date),
quarter(booking_date),
month(booking_date))
select *, 
100*total_cancelled/total_bookings as cancellation_rate
from cr
"""

booking_analytics_df = db.sql(booking_analytics).df()

In [11]:
bucket_name = "airbnb-pipeline-data"
file_key = "gold/booking_analytics.parquet"

buffer = io.BytesIO()
booking_analytics_df.to_parquet(buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name, Key=file_key, Body=buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [15]:
revenue_analytics = """
select year(booking_date) as year,
quarter(booking_date) as quarter,
month(booking_date) as month,
sum(total_amount) as total_revenue,
round(avg(price_per_night),2) as avg_price_per_night
from 
obt
group by year(booking_date),
quarter(booking_date),
month(booking_date)

"""
revenue_analytics_df = db.sql(revenue_analytics).df()

In [17]:
bucket_name = "airbnb-pipeline-data"
file_key = "gold/revenue_analytics.parquet"

buffer = io.BytesIO()
revenue_analytics_df.to_parquet(buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name, Key=file_key, Body=buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [27]:
host_analytics = """
select 
year(booking_date) as year,
quarter(booking_date) as quarter,
month(booking_date) as month,
host_name,
sum(total_amount) as total_revenue,
round(avg(price_per_night),2) as avg_price_per_night,
sum(accommodates) as total_accommodates,
sum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled,
sum(case when accommodation_type='Large' then 1 else 0 end)
as large_accommodates
from obt
group by 
year(booking_date),
quarter(booking_date),
month(booking_date),
host_name
"""

host_analytics_df = db.sql(host_analytics).df()

In [28]:
bucket_name = "airbnb-pipeline-data"
file_key = "gold/host_analytics.parquet"

buffer = io.BytesIO()
host_analytics_df.to_parquet(buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name, Key=file_key, Body=buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [24]:
listing_analytics = """
select 
year(booking_date) as year,
quarter(booking_date) as quarter,
month(booking_date) as month,
listing_id,
count(distinct booking_id) as total_bookings,
sum(total_amount) as total_revenue,
avg(accommodates) avg_accommodates,
sum(case when booking_status = 'cancelled' then 1 else 0 end) AS total_cancelled
from obt
group by 
year(booking_date),
quarter(booking_date),
month(booking_date),
listing_id
"""

listing_analytics_df = db.sql(listing_analytics).df()

In [26]:
bucket_name = "airbnb-pipeline-data"
file_key = "gold/listing_analytics.parquet"

buffer = io.BytesIO()
listing_analytics_df.to_parquet(buffer, index=False)

s3 = boto3.client('s3')

obj = s3.put_object(Bucket=bucket_name, Key=file_key, Body=buffer.getvalue())

print("File successfully uploaded to S3!")

File successfully uploaded to S3!


In [33]:
bucket_name = "airbnb-pipeline-data"
file_key = "gold/listing_analytics.parquet"

s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket_name, Key=file_key)

la = pd.read_parquet(io.BytesIO(obj['Body'].read()))

In [34]:
la

,year,quarter,month,listing_id,total_bookings,total_revenue,avg_accommodates,total_cancelled
0,2024,4,12,379,1,669.0,3.0,0.0
1,2025,1,2,111,3,5041.0,6.0,2.0
2,2025,2,6,253,3,6496.0,1.0,2.0
3,2025,1,2,372,1,733.0,1.0,1.0
4,2025,4,12,425,1,571.0,5.0,1.0
...,...,...,...,...,...,...,...,...
3418,2025,4,11,288,1,827.0,4.0,0.0
3419,2025,1,3,491,1,2501.0,4.0,0.0
3420,2025,4,12,489,1,1651.0,4.0,1.0
3421,2025,3,8,49,1,1595.0,5.0,0.0
